# NB-InvSym 0: Parameter analysis for broken-inversion-symmetry descriptors

**Purpose.** Decide the free parameters of the inversion/mirror descriptors from the actual
geometry of the 200 structures, the way nb0 set Rc and the GRDF Gaussian centres from a geometry scan
instead of guessing. **No descriptors are built here and no parameters are assumed.** This notebook
only measures distributions and prints them.

**Every block is broken down per class, never pooled.** A pooled mean is misleading: if the
nearest-image distance is ~2.5 A for rashba and ~5.0 A for unmatched, the pooled mean (~3.x) describes
neither, and worse, it hides the fact that the classes *separate* - which is itself the signal that the
descriptor works. So each quantity is reported for:
  - the three raw sets:   **rashba (99)**, **dresselhaus (25)**, **unmatched (76)**
  - the classifier target: **split (124 = rashba+dresselhaus)** vs **no-split (76 = unmatched)**
  - **all (200)** only as a reference line, never as the basis for choosing a parameter.

Where a quantity is a core parameter (Block 2 shells, Block 3 nearest-image, Block 6 buckling) we also
print the per-class **cumulative curves side by side** and flag explicitly when the classes separate,
because a separating distribution is a candidate descriptor on its own.

Place this notebook in `Keshav-DDP/Broken-inversion-symmetry/`. It reuses the folder layout and loader
of `nb_expand_dataset.ipynb`.

Output: per-class printed distributions + one CSV (`invsym_param_scan.csv`) you can paste back.

## Cell 1: Config (same paths as nb_expand_dataset)

In [ ]:
import os

# This notebook lives in Broken-inversion-symmetry/, so BASE_DIR is its parent (Keshav-DDP).
BASE_DIR = os.path.abspath("..")

SET_DIRS = {
    "rashba":      os.path.join(BASE_DIR, "Inverse-design", "rashba"),
    "dresselhaus": os.path.join(BASE_DIR, "Inverse-design", "dresselhaus"),
    "unmatched":   os.path.join(BASE_DIR, "Inverse-design", "unmatched"),
}

RESULTS_DIR = os.path.join(".", "invsym-results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("PATH CHECK")
for s, d in SET_DIRS.items():
    print(f"  [{'OK' if os.path.isdir(d) else 'MISSING'}] {s:12s} {d}")

## Cell 2: Imports + load every POSCAR_std (all 200 uids)

Same loader convention as nb_expand_dataset. We walk the three set directories; a uid present in both
rashba and dresselhaus is assigned to rashba (rashba dir scanned first), matching the `dir_for` rule.

In [ ]:
import glob
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from collections import Counter
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element

def load_std(folder):
    p = os.path.join(folder, "POSCAR_std")
    return Structure.from_file(p) if os.path.exists(p) else None

structures = {}   # uid -> Structure
set_of = {}       # uid -> raw set name: rashba / dresselhaus / unmatched
skipped = []
for sname, d in SET_DIRS.items():
    for folder in glob.glob(os.path.join(d, "*")):
        name = os.path.basename(folder)
        idx = name.rfind("-")
        if idx == -1:
            continue
        uid = name[idx + 1:]
        if uid in structures:
            continue
        s = load_std(folder)
        if s is None:
            skipped.append((sname, uid, "no POSCAR_std")); continue
        if max(site.specie.Z for site in s) <= 3:   # the H/He/Li mis-parse guard from nb5
            skipped.append((sname, uid, "maxZ<=3")); continue
        structures[uid] = s
        set_of[uid] = sname

# Two extra class labelings used throughout for the per-class breakdowns.
split_of = {uid: ("no-split" if set_of[uid] == "unmatched" else "split") for uid in structures}

RAW_ORDER   = ["rashba", "dresselhaus", "unmatched"]   # 99 / 25 / 76
SPLIT_ORDER = ["split", "no-split"]                     # 124 / 76

print(f"loaded {len(structures)} structures")
print("by raw set :", {k: dict(Counter(set_of.values())).get(k, 0) for k in RAW_ORDER})
print("by split   :", {k: dict(Counter(split_of.values())).get(k, 0) for k in SPLIT_ORDER})
print("skipped    :", len(skipped), skipped[:6])

## Cell 3: Per-class reporting helpers

`per_class_describe` prints count, mean, and 10/25/50/75/90 percentiles per class (raw sets, split
groups, and an ALL reference row). `per_class_cumulative` prints the fraction below each threshold per
class side by side - how we see separation and the knee. `flag_separation` prints a one-line verdict
on the split vs no-split median gap.

In [ ]:
PCTS = [10, 25, 50, 75, 90]

def per_class_describe(values, labels, title, label_order):
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels, dtype=object)
    print(title)
    hdr = f"  {'class':12s} {'n':>6} {'mean':>8}" + "".join(f"{('p'+str(p)):>8}" for p in PCTS)
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for key in list(label_order) + ["ALL"]:
        v = values if key == "ALL" else values[labels == key]
        if v.size == 0:
            print(f"  {key:12s} {0:6d}"); continue
        ps = np.percentile(v, PCTS)
        print(f"  {key:12s} {v.size:6d} {v.mean():8.3f}" + "".join(f"{x:8.3f}" for x in ps))

def per_class_cumulative(values, labels, thresholds, title, label_order):
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels, dtype=object)
    print(title)
    keys = list(label_order) + ["ALL"]
    print("  thr   " + "".join(f"{k:>12s}" for k in keys))
    for t in thresholds:
        cells = []
        for key in keys:
            v = values if key == "ALL" else values[labels == key]
            cells.append("  --  " if v.size == 0 else f"{(v <= t).mean():.3f}")
        print(f"  {t:5.2f} " + "".join(f"{c:>12s}" for c in cells))

def flag_separation(values, labels, title):
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels, dtype=object)
    a = values[labels == "split"]; b = values[labels == "no-split"]
    if a.size == 0 or b.size == 0:
        print(f"  [sep] {title}: not enough data"); return
    ma, mb = np.median(a), np.median(b)
    rel = abs(ma - mb) / (abs(ma) + abs(mb) + 1e-9)
    verdict = "SEPARATES (candidate descriptor)" if rel > 0.15 else "weak/none"
    print(f"  [sep] {title}: split median={ma:.3f}  no-split median={mb:.3f}  rel gap={rel:.2f}  -> {verdict}")

print("helpers ready")

## Cell 4: Block 1 - cell-size families, atom counts, z-extent (per class)

Confirms cell-size families and slab geometry, split by class. If the unmatched set is systematically
thinner or smaller, that is both a confound to note and a hint that buckling-type descriptors carry
class signal.

In [ ]:
rows = []
for uid, s in structures.items():
    cart = s.cart_coords; abc = s.lattice.abc
    vac_axis = int(np.argmax(abc)); z = cart[:, vac_axis]
    rows.append(dict(uid=uid, set=set_of[uid], split=split_of[uid], natoms=len(s),
                     nelem=len(set(site.specie.symbol for site in s)),
                     vac_axis=vac_axis, z_spread=float(z.max() - z.min())))
info = pd.DataFrame(rows)

print("ATOM COUNT per class (value_counts):")
for key in RAW_ORDER:
    vc = info.loc[info["set"] == key, "natoms"].value_counts().sort_index()
    print(f"  {key:12s} {vc.to_dict()}")
print("\nVACUUM AXIS per class:")
for key in RAW_ORDER:
    print(f"  {key:12s} {info.loc[info['set']==key,'vac_axis'].value_counts().to_dict()}")
print()
per_class_describe(info["z_spread"].values, info["set"].values,
                   "Z-SPREAD (slab thickness, Angstrom) by raw set:", RAW_ORDER)
print()
per_class_describe(info["z_spread"].values, info["split"].values, "Z-SPREAD by split group:", SPLIT_ORDER)
print()
per_class_describe(info["natoms"].values.astype(float), info["split"].values,
                   "ATOM COUNT by split group (watch for size confound):", SPLIT_ORDER)

## Cell 5: Block 2 - neighbour shell structure vs radius (per class)

Neighbour counts at candidate cutoffs, per class, plus the radial histogram per class. If the shell
structure differs by class we pick Rc_inv to capture the 1st+2nd shell for *all* classes, reading the
counts per class rather than from a pooled mean.

In [ ]:
CANDIDATE_RC = [3.0, 3.5, 4.0, 4.5, 5.0, 6.0]
RC_MAX = max(CANDIDATE_RC)

atom_set_label = []
count_at = {rc: [] for rc in CANDIDATE_RC}
first_dist = []; first_dist_set = []
dist_by_set = {k: [] for k in RAW_ORDER}
for uid, s in structures.items():
    alln = s.get_all_neighbors(RC_MAX)
    for i in range(len(s)):
        dd = np.array([nb.nn_distance for nb in alln[i] if nb.nn_distance > 1e-3])
        atom_set_label.append(set_of[uid])
        if dd.size == 0:
            for rc in CANDIDATE_RC: count_at[rc].append(0)
            continue
        first_dist.append(dd.min()); first_dist_set.append(set_of[uid])
        dist_by_set[set_of[uid]].extend(dd.tolist())
        for rc in CANDIDATE_RC:
            count_at[rc].append(int((dd <= rc).sum()))
atom_set_label = np.array(atom_set_label, dtype=object)

print("MEAN neighbours per atom at each Rc, per raw set:")
print(f"  {'Rc':>5}" + "".join(f"{k:>13s}" for k in RAW_ORDER + ['ALL']))
for rc in CANDIDATE_RC:
    c = np.array(count_at[rc], dtype=float)
    vals = [c[atom_set_label == k].mean() if (atom_set_label == k).any() else np.nan for k in RAW_ORDER]
    print(f"  {rc:5.1f}" + "".join(f"{v:13.1f}" for v in vals) + f"{c.mean():13.1f}")

print()
per_class_describe(np.array(first_dist), np.array(first_dist_set, dtype=object),
                   "NEAREST-NEIGHBOUR distance (Angstrom) by raw set:", RAW_ORDER)

print("\nALL-neighbour-distance histogram per raw set (shell structure), % within each set:")
bins = np.arange(1.5, RC_MAX + 0.5, 0.5)
print("  bin        " + "".join(f"{k:>13s}" for k in RAW_ORDER))
hists = {k: np.histogram(np.array(dist_by_set[k]), bins=bins)[0] for k in RAW_ORDER}
tot = {k: max(hists[k].sum(), 1) for k in RAW_ORDER}
for bi in range(len(bins) - 1):
    cells = "".join(f"{100*hists[k][bi]/tot[k]:13.1f}" for k in RAW_ORDER)
    print(f"  {bins[bi]:4.1f}-{bins[bi+1]:4.1f}" + cells)

## Cell 6: Block 3 - nearest inversion-image distance (per class; sets the pairing tolerance)

The core scan. For each atom *i* (a candidate **local** inversion centre) and each neighbour vector
**v**, the inversion image sits at **-v**; we record the distance to the nearest actual atom there, for
ANY species and for SAME species. Full per-class distribution + per-class cumulative curve, so if
rashba sits at ~2.5 A and unmatched at ~5.0 A we see it directly instead of averaging to a meaningless
middle. The pairing tolerance is read from where the curves knee, chosen to hold across classes.

In [ ]:
WORK_RC = 5.0   # provisional measurement radius only; final Rc_inv chosen from Block 2 per-class counts

def neighbor_vectors(s, rc):
    alln = s.get_all_neighbors(rc)
    out = []
    for i, site in enumerate(s):
        ci = site.coords; v, sym = [], []
        for nb in alln[i]:
            if nb.nn_distance <= 1e-3:
                continue
            v.append(np.array(nb.coords) - ci); sym.append(nb.specie.symbol)
        out.append(dict(v=np.array(v) if v else np.zeros((0, 3)), sym=np.array(sym, dtype=object)))
    return out

rec_any, rec_same = [], []
lab_any_set, lab_any_split = [], []
lab_same_set = []
no_same = {k: 0 for k in RAW_ORDER}; n_pairs = {k: 0 for k in RAW_ORDER}
for uid, s in structures.items():
    sk, spk = set_of[uid], split_of[uid]
    for a in neighbor_vectors(s, WORK_RC):
        V = a["v"]; sym = a["sym"]
        if len(V) < 2:
            continue
        for j in range(len(V)):
            n_pairs[sk] += 1
            dd = np.linalg.norm(V - (-V[j]), axis=1)
            rec_any.append(dd.min()); lab_any_set.append(sk); lab_any_split.append(spk)
            m = (sym == sym[j])
            if m.any():
                rec_same.append(dd[m].min()); lab_same_set.append(sk)
            else:
                no_same[sk] += 1

print("(atom,neighbour) pairs with NO same-species candidate in range, per set:")
for k in RAW_ORDER:
    print(f"  {k:12s} {no_same[k]:7d} / {n_pairs[k]:7d}  ({100*no_same[k]/max(n_pairs[k],1):.1f}%)")

print()
per_class_describe(rec_any, lab_any_set, "NEAREST-IMAGE distance, ANY species, by raw set:", RAW_ORDER)
print()
per_class_describe(rec_any, lab_any_split, "NEAREST-IMAGE distance, ANY species, by split group:", SPLIT_ORDER)
print()
per_class_describe(rec_same, lab_same_set, "NEAREST-IMAGE distance, SAME species, by raw set:", RAW_ORDER)

TOLS = [0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
print()
per_class_cumulative(rec_any, lab_any_set, TOLS,
                     "CUMULATIVE fraction with nearest-image (ANY) <= tol, by raw set (find the knee):", RAW_ORDER)
print()
per_class_cumulative(rec_any, lab_any_split, TOLS,
                     "CUMULATIVE fraction with nearest-image (ANY) <= tol, by split group:", SPLIT_ORDER)
print()
flag_separation(rec_any, lab_any_split, "nearest-image (ANY) per-pair")

## Cell 7: Block 4 - per-STRUCTURE inversion residual at local centres (per class separation check)

Aggregates Block 3 to one scalar per structure - the median, mean, AND max over atoms (you want both
mean- and max-type aggregation) of the per-atom median nearest-image distance. Reports the full
per-class distribution and the split-vs-no-split verdict. Directional check that the operation carries
structure-level class signal before building the full family. Per-structure values saved to CSV.

In [ ]:
per_struct = []
for uid, s in structures.items():
    atom_med = []
    for a in neighbor_vectors(s, WORK_RC):
        V = a["v"]
        if len(V) < 2:
            continue
        leftovers = [np.linalg.norm(V - (-V[j]), axis=1).min() for j in range(len(V))]
        atom_med.append(np.median(leftovers))
    if atom_med:
        per_struct.append(dict(uid=uid, set=set_of[uid], split=split_of[uid],
                               inv_resid_median=float(np.median(atom_med)),
                               inv_resid_mean=float(np.mean(atom_med)),
                               inv_resid_max=float(np.max(atom_med))))
ps = pd.DataFrame(per_struct)

for col in ["inv_resid_median", "inv_resid_mean", "inv_resid_max"]:
    per_class_describe(ps[col].values, ps["set"].values, f"{col} by raw set:", RAW_ORDER)
    print()
for col in ["inv_resid_median", "inv_resid_mean", "inv_resid_max"]:
    per_class_describe(ps[col].values, ps["split"].values, f"{col} by split group:", SPLIT_ORDER)
    flag_separation(ps[col].values, ps["split"].values, col)
    print()

ps.to_csv(os.path.join(RESULTS_DIR, "invsym_param_scan.csv"), index=False)
print(f"saved {os.path.join(RESULTS_DIR, 'invsym_param_scan.csv')}")

## Cell 8: Block 5 - global best-fit inversion-centre search box + resolution (per class)

For the optimal-centre variant: in-plane fractional extent (search-box size) per class, and the
residual profiled at three grid resolutions on a few structures per class, so the grid step resolves
the minimum for all classes, not just one.

In [ ]:
frac_extent_set = {k: [] for k in RAW_ORDER}
for uid, s in structures.items():
    frac_extent_set[set_of[uid]].append(s.frac_coords.ptp(axis=0))
print("median fractional-coord spread per axis (ptp), per raw set:")
for k in RAW_ORDER:
    if frac_extent_set[k]:
        print(f"  {k:12s} {np.median(np.array(frac_extent_set[k]), axis=0).round(3)}")
print("=> a centre search over frac in [0,0.5] per in-plane axis covers all symmetry-distinct centres")

def mean_residual_for_center(s, center_cart):
    cart = s.cart_coords; tot = 0.0
    for i in range(len(s)):
        img = 2 * center_cart - cart[i]
        tot += np.linalg.norm(cart - img, axis=1).min()
    return tot / max(len(s), 1)

samples = []
for tgt in RAW_ORDER:
    got = 0
    for uid in structures:
        if set_of[uid] == tgt:
            samples.append(uid); got += 1
        if got == 2:
            break

for uid in samples:
    s = structures[uid]; L = s.lattice
    print(f"\n{set_of[uid]:11s} {uid}  (natoms={len(s)})")
    for g in [0.25, 0.125, 0.0625]:
        nlin = int(round(0.5 / g)) + 1; best = np.inf
        for fa in np.linspace(0, 0.5, nlin):
            for fb in np.linspace(0, 0.5, nlin):
                best = min(best, mean_residual_for_center(s, L.get_cartesian_coords([fa, fb, 0.5])))
        print(f"    grid step {g:.4f} ({nlin}x{nlin}) -> best mean residual {best:.4f}")

## Cell 9: Block 6 - buckling baseline and electronegativity spread (per class)

For the z-projected and EN-weighted variants. Per class: heavy-vs-light sublattice z-separation
(buckling proxy) and electronegativity range across species. If these separate by class they are
candidate descriptors; if near zero for everyone they are not resolvable and we drop them.

In [ ]:
rows = []
for uid, s in structures.items():
    abc = s.lattice.abc; vac_axis = int(np.argmax(abc))
    zc = s.cart_coords[:, vac_axis]
    syms = np.array([site.specie.symbol for site in s]); Zs = np.array([site.specie.Z for site in s])
    Xvals = [Element(el).X for el in set(syms) if Element(el).X is not None]
    en_range = (max(Xvals) - min(Xvals)) if len(Xvals) > 1 else 0.0
    heavy = syms == syms[np.argmax(Zs)]; light = syms == syms[np.argmin(Zs)]
    buck = abs(zc[heavy].mean() - zc[light].mean()) if (heavy.any() and light.any()) else 0.0
    rows.append(dict(set=set_of[uid], split=split_of[uid], en_range=en_range, buckling=buck))
bf = pd.DataFrame(rows)

per_class_describe(bf["buckling"].values, bf["set"].values,
                   "HEAVY-vs-LIGHT sublattice z-separation (buckling proxy, A) by raw set:", RAW_ORDER)
print()
per_class_describe(bf["buckling"].values, bf["split"].values, "buckling proxy by split group:", SPLIT_ORDER)
flag_separation(bf["buckling"].values, bf["split"].values, "buckling proxy")
print()
per_class_describe(bf["en_range"].values, bf["set"].values,
                   "ELECTRONEGATIVITY range across species (Pauling) by raw set:", RAW_ORDER)
print()
per_class_describe(bf["en_range"].values, bf["split"].values, "EN range by split group:", SPLIT_ORDER)
flag_separation(bf["en_range"].values, bf["split"].values, "EN range")
print("\nbuckling proxy: fraction > 0.1 A and > 0.5 A, per raw set:")
for k in RAW_ORDER:
    v = bf.loc[bf["set"] == k, "buckling"].values
    if v.size:
        print(f"  {k:12s} >0.1A: {(v>0.1).mean():.2f}   >0.5A: {(v>0.5).mean():.2f}")

## What I need from the output

Paste back the printed output of Blocks 1-6. Because everything is per class, I read both the parameter
AND whether the quantity is already a descriptor:
- **Rc_inv** from Block 2 (smallest radius capturing the 1st+2nd shell for every class),
- **pairing tolerance** from Block 3 (the knee in the per-class cumulative curves, holding across classes),
- **same-species vs any-species** from Block 3 (per-set % with no same-species partner),
- **global-centre grid step + box** from Block 5 (coarsest grid resolving the minimum for all classes),
- whether **buckling and EN-weighting** are worth including from Block 6,
- and any `[sep] ... SEPARATES (candidate descriptor)` flags - those go straight into the descriptor
  notebook, since a class-separating distribution is exactly the broken-inversion signal ma'am wants.

Then I build `nb_invsym_01_descriptors.ipynb` (local / global / origin centre x inversion / mirror,
plain / sqrt(Zi*Zj) / Z^4, distance + angle residuals, z-projection, buckling, top/bottom asymmetry),
with mean AND max aggregation, merged into `work` and tested on the 99 / 124 / 200 splits against both
the regression and the splits-vs-no-split classifier.